In [ ]:
# Chẩn đoán môi trường Kaggle (Phase 5)
import subprocess
import sys
from pathlib import Path


def run(cmd):
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        return result.stdout + result.stderr
    except Exception as exc:
        return f"KHÔNG CHẠY ĐƯỢC: {exc}"


print("=" * 70)
print("1. Python")
print("=" * 70)
print(sys.version)

print("=" * 70)
print("2. GPU")
print("=" * 70)
try:
    import torch

    print("torch:", torch.__version__)
    print("torch.version.cuda:", torch.version.cuda)
    print("cuda.is_available():", torch.cuda.is_available())
    print("device_count():", torch.cuda.device_count())
    if torch.cuda.is_available():
        print("Tên GPU:", torch.cuda.get_device_name(0))
        print(
            "Dung lượng (GB):",
            round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        )
except Exception as exc:
    print("LỖI torch:", exc)

print("=" * 70)
print("3. nvidia-smi")
print("=" * 70)
print(run(["nvidia-smi"])[:1200])

print("=" * 70)
print("4. Thiết bị / biến môi trường")
print("=" * 70)
print(run(["ls", "-la", "/dev/nvidia*"])[:600])
for key in ("KAGGLE_KERNEL_RUN_TYPE", "KAGGLE_GPU", "CUDA_VISIBLE_DEVICES"):
    print(f"{key} = {__import__('os').environ.get(key)}")

print("=" * 70)
print("5. Internet / pip")
print("=" * 70)
for url in ["https://pypi.org/simple/", "https://huggingface.co"]:
    print(f"{url} ->", run(["curl", "-sS", "-o", "/dev/null", "-w", "%{http_code}", "--max-time", "20", url]).strip()[:200])

print("=" * 70)
print("6. Thư viện quan trọng")
print("=" * 70)
for module in ["transformers", "accelerate", "numpy", "sklearn", "pandas", "tokenizers", "sentencepiece"]:
    try:
        imported = __import__(module)
        print(f"{module}: {getattr(imported, '__version__', '?')}")
    except Exception as exc:
        print(f"{module}: THIẾU ({exc})")

print("=" * 70)
print("7. Dataset mount")
print("=" * 70)
input_root = Path("/kaggle/input")
try:
    for path in sorted(input_root.rglob("*")):
        if path.is_file() and path.stat().st_size > 0:
            print("  ", path, f"{path.stat().st_size / 1024 / 1024:.1f} MB")
except Exception as exc:
    print("LỖI:", exc)

print("=" * 70)
print("8. Tải model từ HuggingFace")
print("=" * 70)
try:
    from huggingface_hub import snapshot_download

    path = snapshot_download("vinai/phobert-base", allow_patterns=["*.json", "*.txt"])
    print("Đã tải:", path)
    print("Nội dung:", sorted(p.name for p in Path(path).iterdir()))
except Exception as exc:
    print("LỖI tải model:", exc)

print("=" * 70)
print("9. pip install torch CUDA (nếu cần)")
print("=" * 70)
print(run(["pip", "download", "--no-deps", "-d", "/tmp/pipcheck", "sentencepiece"])[:600])

print("DIAGNOSIS_DONE")
